# 00 — Carga y EDA

**Objetivo.** Cargar todos los CSV, diagnosticar shapes/fechas/NaN, visualizar las 6 series y ejecutar el DETECTIVE: identificar el índice fuente y lag de D (Ghost), correlación macro↔C, y correlación network↔F. Las decisiones de este notebook son el input estratégico de todos los demás.

**Decisión tomada.** Notebook único de EDA ejecutado primero por el equipo completo. Sin él no se puede fijar la estrategia de D ni los features auxiliares de C/F.

**Qué hace.** Carga, shapes, fechas, NaN report, plot de las 6 series, estadísticas descriptivas de volatilidad, `lagged_correlation` para Ghost, correlaciones macro↔C y network↔F.

**Qué NO hace.** No entrena nada, no predice, no escribe ficheros en `results/` ni `models/`.

**Inputs.** `data/train_indices.csv`, `data/test_dates.csv`, `data/train_news.csv`, `data/test_news.csv`, `data/train_macro_factors.csv`, `data/test_macro_factors.csv`, `data/train_network_metrics.csv`, `data/test_network_metrics.csv`

**Output esperado.** Celdas de diagnóstico rellenadas con los hallazgos. Ningún fichero escrito en disco.

## 0. GPU workaround + imports

La RTX 5070 Ti (Blackwell) es incompatible con TensorFlow GPU. Esta línea va ANTES de cualquier import de TF/Keras.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    load_hackathon_data, lagged_correlation,
    precios_a_logret,
    DATA_DIR, VAL_DAYS, V_IN_SHARED, INDEX_COLS
)

## 1. Carga de datos

`load_hackathon_data` es robusta a ficheros ausentes: avisa con warning pero no falla. Útil antes de que el profesor entregue todos los CSV.

In [ ]:
data = load_hackathon_data(DATA_DIR)
idx  = data['train_indices']   # DataFrame de precios de los 6 índices

# Mostrar qué ficheros se cargaron
print('Ficheros disponibles:', list(data.keys()))

## 2. Diagnóstico básico: fechas, longitud, NaN

Primero ver cuántos datos hay. Si las series son cortas (< ~1500 días), revisar si V_IN_SHARED=20 es excesivo.

In [ ]:
# Shapes y fechas de cada DataFrame cargado
for key, df in data.items():
    try:
        rng = f"{df.index[0].date()} -> {df.index[-1].date()}"
    except Exception:
        rng = f"{df.index[0]} -> {df.index[-1]}"
    print(f"{key:<20} {str(df.shape):<15} {rng}")

# NaN por columna en train_indices
print("\nNaN en train_indices:")
print(idx.isnull().sum())

## 3. Visualización de las 6 series (precios brutos)

Ver escala, tendencia y volatilidad relativa. Las diferencias de escala entre índices confirman por qué operamos en log-retornos.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), INDEX_COLS):
    if col in idx.columns:
        ax.plot(idx[col].dropna().values, lw=0.8)
        ax.set_title(col)
        ax.set_xlabel('días')
plt.suptitle('Precios brutos — train_indices', y=1.01)
plt.tight_layout()
plt.show()

## 4. Estadísticas descriptivas de volatilidad

La volatilidad diaria (std de log-retornos) y el drift determinan qué índices dominan el RMSE y qué baselines tienen sentido.

In [ ]:
# Calcular log-retornos de cada índice y resumir
stats = {}
for col in INDEX_COLS:
    if col not in idx.columns:
        continue
    rets = precios_a_logret(idx[col].dropna().values)
    stats[col] = {
        'vol_diaria': rets.std(),
        'drift_diario': rets.mean(),
        'min_precio': idx[col].min(),
        'max_precio': idx[col].max(),
    }

pd.DataFrame(stats).T.round(6)

## 5. DETECTIVE — Ghost (Index D)

La pista del enunciado dice que D sigue con lag una señal oculta en otro índice. `lagged_correlation` busca la correlación de D con cada columna candidata a lags 0..30. El pico de correlación absoluta revela el índice fuente y el lag.

**Nota de firma:** `lagged_correlation(df, target_col, candidate_cols=None, max_lag=30)` — `df` es un DataFrame con todas las series. Operar sobre log-retornos (no precios) para eliminar tendencias espurias.

In [ ]:
# Convertir a log-retornos para eliminar tendencias
# log_idx: DataFrame de log-retornos diarios de todos los índices
log_idx = np.log(idx).diff().dropna()

# Correlaciones de Index_D con todos los demás a lags 0..30
df_corr = lagged_correlation(log_idx, target_col='Index_D', max_lag=30)

# Heatmap de correlaciones por lag
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(df_corr.T.values, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set_yticks(range(len(df_corr.columns)))
ax.set_yticklabels(df_corr.columns)
ax.set_xlabel('Lag (días)')
ax.set_title('Correlación de Index_D con candidatos — por lag')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Lag y columna de correlación máxima
max_corr_by_col = df_corr.abs().idxmax()   # lag óptimo por columna candidata
print("Lag óptimo por columna candidata:\n", max_corr_by_col)
print("\nCorrelación máxima global:", df_corr.abs().stack().idxmax())

### Decisión Ghost — RELLENAR TRAS EJECUTAR

- **Índice fuente:** `?`  
- **Lag:** `?` días  
- **Correlación:** `?`  

Trasladar estos valores a `06_index_D.ipynb`.

In [ ]:
# Confirmación visual: D vs índice_fuente desplazado por lag
# Rellenar tras la celda anterior
GHOST_SOURCE = 'Index_?'   # <-- rellenar con el índice fuente encontrado
GHOST_LAG    = 0           # <-- rellenar con el lag encontrado

if GHOST_SOURCE in log_idx.columns and GHOST_LAG > 0:
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(log_idx['Index_D'].values, label='Index_D', lw=0.8)
    ax.plot(log_idx[GHOST_SOURCE].shift(GHOST_LAG).values,
            label=f'{GHOST_SOURCE} (lag={GHOST_LAG})', lw=0.8, ls='--')
    ax.set_title('Confirmación visual: D vs fuente desplazada')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. DETECTIVE — Energy (Index C) ↔ macro_factors

La pista dice que C está ligado a energía global y macro. Buscar correlación con las columnas de `train_macro_factors` (oro, crudo, tipos).

In [ ]:
if 'train_macro' in data:
    macro = data['train_macro']
    print('Columnas macro:', list(macro.columns))

    # Correlaciones entre log-ret de C y log-ret de cada columna macro a distintos lags
    # Unir C y macro en un DataFrame alineado por fecha
    combined = pd.concat([
        log_idx[['Index_C']],
        np.log(macro.replace(0, np.nan)).diff().dropna()
    ], axis=1).dropna()

    macro_cols = [c for c in combined.columns if c != 'Index_C']
    df_corr_c  = lagged_correlation(combined, target_col='Index_C',
                                    candidate_cols=macro_cols, max_lag=20)
    print("\nCorrelaciones macro↔C por lag:")
    print(df_corr_c.round(3))
else:
    print('[AVISO] train_macro no disponible todavía')

### Decisión macro-C — RELLENAR TRAS EJECUTAR

- **Features de macro seleccionadas:** `[?, ?, ...]`  
- **Lag:** `?` días (si aplica)  

Trasladar a `05_index_C.ipynb`.

## 7. DETECTIVE — Digital (Index F) ↔ network_metrics

La pista dice que F está atado a métricas on-chain de `network_metrics` (node counts, transaction volume, etc.).

In [ ]:
if 'train_network' in data:
    net = data['train_network']
    print('Columnas network:', list(net.columns))

    # Correlaciones entre log-ret de F y columnas de network a distintos lags
    combined_f = pd.concat([
        log_idx[['Index_F']],
        net.diff().dropna()   # diferencias de las métricas on-chain
    ], axis=1).dropna()

    net_cols  = [c for c in combined_f.columns if c != 'Index_F']
    df_corr_f = lagged_correlation(combined_f, target_col='Index_F',
                                   candidate_cols=net_cols, max_lag=20)
    print("\nCorrelaciones network↔F por lag:")
    print(df_corr_f.round(3))
else:
    print('[AVISO] train_network no disponible todavía')

### Decisión network-F — RELLENAR TRAS EJECUTAR

- **Features de network seleccionadas:** `[?, ?, ...]`  
- **Lag:** `?` días (si aplica)  

Trasladar a `08_index_F.ipynb`.

## 8. RESUMEN DE DECISIONES ESTRATÉGICAS

Rellenar después de ejecutar todo el notebook. Este bloque es la referencia para el equipo.

| Índice | Approach decidido | Justificación |
|--------|-------------------|---------------|
| A | LSTM + ensemble | alta vol, domina RMSE |
| B | baseline_flat | defensivo, confirmar en 04 |
| C | LSTM + macro | correlación con: `?` |
| D | ghost | fuente=`?`, lag=`?` |
| E | LSTM o baseline | confirmar en 07 |
| F | LSTM + network | correlación con: `?` |

**V_IN_SHARED acordado:** `20` (provisional — ajustar si series cortas)